# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nadim7009/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

**Lane 4, CTR opportunity scoring.** One row is one page. The target is the page's observed `ctr`. The feature vector has six columns, all measured in the same 90-day window as the target, or fixed traits of the page:

`avg_position`, `impressions_90d`, `word_count`, `content_age_days` (numbers) and `content_type`, `main_intent` (categories).

Handling: missing categories become the label `unknown` and are kept as their own category, because "no intent assigned" is itself something the model can see. Missing `word_count` is left as missing, since the boosting model handles it natively and a made-up fill value (zero, the median) would invent information.

In [1]:
# Setup: works in Colab (clones my repo) and locally (walks up to the repo root)
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/nadim7009/flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold, KFold
from sklearn.ensemble import HistGradientBoostingRegressor
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
pool = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 500)].reset_index(drop=True)
print(f"{len(df):,} pages in the file; {len(pool):,} in my lane's pool from {pool['client_id'].nunique()} clients")

FEATS = ["avg_position", "impressions_90d", "word_count", "content_age_days", "content_type", "main_intent"]
X = pool[FEATS].copy()
for col in ["content_type", "main_intent"]:
    X[col] = X[col].fillna("unknown").astype("category")
y = pool["ctr"]
print("Feature vector:", X.shape, "| target: ctr (percent)")
print(X.dtypes.to_string())
X.head()

30,000 pages in the file; 12,023 in my lane's pool from 28 clients
Feature vector: (12023, 6) | target: ctr (percent)
avg_position         float64
impressions_90d        int64
word_count           float64
content_age_days       int64
content_type        category
main_intent         category


,avg_position,impressions_90d,word_count,content_age_days,content_type,main_intent
0,10.6,3803,3221.0,187,keyword article,transactional
1,6.2,11751,NaN,463,keyword article,commercial
2,8.5,3970,3080.0,147,keyword article,transactional
3,4.9,1240,NaN,257,keyword article,informational
4,2.2,20919,NaN,329,keyword article,commercial


## 2. Feature notes (meaning, missing, categorical, available-when?)

The last column is the one that matters for leakage: does this value exist **before** the moment someone uses the score?

In [2]:
notes = pd.DataFrame([
    ["avg_position",    "mean rank in search results over 90d",  "none missing",  "number",      "yes, measured in the window being scored"],
    ["impressions_90d", "times the page was shown",               "none missing",  "number",      "yes; floor of 500 keeps CTR from swinging on one click"],
    ["word_count",      "length of the page text",                "left missing",  "number",      "yes, a fixed trait of the page"],
    ["content_age_days","days since the page was created",        "none missing",  "number",      "yes"],
    ["content_type",    "kind of article",                        "'unknown'",     "category",    "yes, a fixed trait"],
    ["main_intent",     "search intent assigned to the page",     "'unknown'",     "category",    "yes, a fixed trait"],
], columns=["feature", "meaning", "missing handled by", "type", "known before scoring?"])
notes["share_missing_in_pool"] = [pool[f].isna().mean().round(3) for f in notes["feature"]]
print(notes.to_string(index=False))

         feature                              meaning missing handled by     type                                  known before scoring?  share_missing_in_pool
    avg_position mean rank in search results over 90d       none missing   number               yes, measured in the window being scored                  0.000
 impressions_90d             times the page was shown       none missing   number yes; floor of 500 keeps CTR from swinging on one click                  0.000
      word_count              length of the page text       left missing   number                         yes, a fixed trait of the page                  0.305
content_age_days      days since the page was created       none missing   number                                                    yes                  0.000
    content_type                      kind of article          'unknown' category                                     yes, a fixed trait                  0.000
     main_intent   search intent assigne

## 3. The leakage hunt

I attack my own features four ways: (a) is any column a rearrangement of the target, (b) is any column built from a later window, (c) can the model learn the client instead of the pattern, (d) does the file hold anything private.

**(a) and (b) are the two real hits in this file.** `clicks_90d` and `ctr` are the same quantity (CTR is clicks over impressions). And `trend_pct` and `trend_direction`, the starter label's source, are built from the last 30 days against the previous 30 days.

In [3]:
# (a) clicks is the target in disguise
recon = pool["clicks_90d"] / pool["impressions_90d"] * 100
print(f"(a) corr(ctr, clicks/impressions*100) = {np.corrcoef(pool['ctr'], recon)[0, 1]:.4f}  -> clicks_90d cannot be a feature")

# (b) the trend fields are an impressions change between two later windows
x = df[df["impressions_prev_30d"] > 0]
change = (x["impressions_last_30d"] - x["impressions_prev_30d"]) / x["impressions_prev_30d"] * 100
print(f"(b) corr(trend_pct, last30-vs-prev30 impressions change) = {np.corrcoef(x['trend_pct'].fillna(0), change)[0, 1]:.4f}"
      "  -> trend_pct IS the label's source")

# What a leak does to a held-out-client score
W = pool["impressions_90d"].values
def held_out_error(cols):
    Xc = pool[cols].copy()
    for col in Xc.columns:
        if not pd.api.types.is_numeric_dtype(Xc[col]): Xc[col] = Xc[col].fillna("unknown").astype("category")
    errs = []
    for tr, te in GroupKFold(5).split(Xc, y, pool["client_id"]):
        m = HistGradientBoostingRegressor(loss="absolute_error", max_iter=200, learning_rate=0.05,
                                          categorical_features="from_dtype", random_state=42)
        m.fit(Xc.iloc[tr], y.iloc[tr], sample_weight=W[tr])
        errs.append(np.average(np.abs(y.iloc[te] - m.predict(Xc.iloc[te])), weights=W[te]))
    return float(np.mean(errs))

clean = held_out_error(FEATS)
leaky = held_out_error(FEATS + ["clicks_90d"])
print(f"\nWeighted CTR error on held-out clients: clean {clean:.3f}   with clicks_90d added {leaky:.3f}")
print("A drop that large is a leak, not skill.")
assert leaky < 0.3 * clean

(a) corr(ctr, clicks/impressions*100) = 1.0000  -> clicks_90d cannot be a feature
(b) corr(trend_pct, last30-vs-prev30 impressions change) = 1.0000  -> trend_pct IS the label's source

Weighted CTR error on held-out clients: clean 0.245   with clicks_90d added 0.046
A drop that large is a leak, not skill.


In [4]:
# (c) client_id: does a random split reward memorising the site?
pool["client_id"] = pool["client_id"].astype(str)
Xc = X.copy(); Xc["client_id"] = pool["client_id"].astype("category")
def err(Xm, splitter, groups=None):
    out = []
    for tr, te in splitter.split(Xm, y, groups):
        m = HistGradientBoostingRegressor(loss="absolute_error", max_iter=200, learning_rate=0.05,
                                          categorical_features="from_dtype", random_state=42)
        m.fit(Xm.iloc[tr], y.iloc[tr], sample_weight=W[tr])
        out.append(np.average(np.abs(y.iloc[te] - m.predict(Xm.iloc[te])), weights=W[te]))
    return float(np.mean(out))
rnd = KFold(5, shuffle=True, random_state=42)
print(f"(c) random split: without client_id {err(X, rnd):.3f}   with client_id {err(Xc, rnd):.3f}")
print("    The error improves when the model is told the client. That is memorising a site, so client_id stays out.")

# (d) privacy scan: any URL-like or domain-like text in string columns?
text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
hits = {c: int(df[c].astype(str).str.contains(r"http|www\.|\.com|\.org", regex=True).sum()) for c in text_cols}
print(f"\n(d) URL/domain-like strings found per text column: {sum(hits.values())} in total")
print("    client_id and content_id are hashed, so no client names or page addresses are present.")
assert sum(hits.values()) == 0

(c) random split: without client_id 0.194   with client_id 0.186
    The error improves when the model is told the client. That is memorising a site, so client_id stays out.

(d) URL/domain-like strings found per text column: 0 in total
    client_id and content_id are hashed, so no client names or page addresses are present.


**Result (measured).** The six chosen features pass. Two obvious columns would have leaked, and one innocent-looking column (`client_id`) would have flattered a random split. The file holds no URLs, domains or query text.

## 4. What I excluded and why

In [5]:
excluded = pd.DataFrame([
    ["clicks_90d, sessions_90d, pageviews_90d, engaged_sessions_90d", "outcome-side numbers; clicks is the target's numerator"],
    ["ctr (as an input)", "it is the target"],
    ["trend_direction, trend_pct", "built from the last-30 vs previous-30 window; the starter label's source"],
    ["impressions_last_30d, impressions_prev_30d (and clicks/sessions versions)", "the same later-window numbers the trend fields are made from"],
    ["client_id", "lets the model memorise a site under a random split; kept only as the grouping key"],
    ["content_id", "an anonymised row key with no meaning"],
    ["search_volume, competition, cpc", "heavily missing (about 8% of pages) and not needed for this lane; left out to keep the vector small"],
    ["provider_used, model_used", "70% missing (provider) and tool metadata, not a page trait"],
], columns=["excluded", "why"])
print(excluded.to_string(index=False))

                                                                 excluded                                                                                                 why
            clicks_90d, sessions_90d, pageviews_90d, engaged_sessions_90d                                              outcome-side numbers; clicks is the target's numerator
                                                        ctr (as an input)                                                                                    it is the target
                                               trend_direction, trend_pct                            built from the last-30 vs previous-30 window; the starter label's source
impressions_last_30d, impressions_prev_30d (and clicks/sessions versions)                                        the same later-window numbers the trend fields are made from
                                                                client_id                  lets the model memorise a site under a 

## Self-check

- [x] Every section above is filled, with the thinking in markdown and the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime, then Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit the repo URL on the card.